In [ ]:


import os
import sys
import math
import time
import random
import pickle
import numpy as np
from collections import defaultdict, deque
from typing import Optional, Tuple, List, Dict

# ── Try importing chess (python-chess) ──────────────────────────────────────
try:
    import chess
    import chess.pgn
except ImportError:
    print("Installing python-chess...")
    os.system("pip install python-chess --quiet")
    import chess

# ── Try importing PyTorch ───────────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    print("Installing PyTorch (CPU)...")
    os.system("pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet")
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        import torch.optim as optim
        from torch.utils.data import DataLoader, TensorDataset
        TORCH_AVAILABLE = True
    except:
        TORCH_AVAILABLE = False
        print("WARNING: PyTorch unavailable, using NumPy fallback engine.")


# ═══════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════

PIECE_CHANNELS = 12        # 6 piece types × 2 colors
EXTRA_CHANNELS = 6         # castling rights × 4, en passant file, turn
TOTAL_CHANNELS = PIECE_CHANNELS + EXTRA_CHANNELS   # 18
BOARD_SIZE = 8
ACTION_SIZE = 4672         # max possible UCI moves in chess (upper bound)
MODEL_PATH = "chess_rl_model.pt"
REPLAY_PATH = "chess_replay.pkl"

PIECE_TO_CH = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}

# Build a consistent move→index mapping
def _build_move_index() -> Dict[str, int]:
    """Enumerate all possible UCI moves for 8×8 board."""
    idx = 0
    mapping = {}
    files = 'abcdefgh'
    ranks = '12345678'
    promos = ['q', 'r', 'b', 'n']
    for f1 in files:
        for r1 in ranks:
            for f2 in files:
                for r2 in ranks:
                    if f1 == f2 and r1 == r2:
                        continue
                    uci = f"{f1}{r1}{f2}{r2}"
                    mapping[uci] = idx
                    idx += 1
                    # promotions from rank 7→8 or 2→1
                    if (r1 == '7' and r2 == '8') or (r1 == '2' and r2 == '1'):
                        for p in promos:
                            mapping[uci + p] = idx
                            idx += 1
    return mapping

MOVE_TO_IDX = _build_move_index()
IDX_TO_MOVE = {v: k for k, v in MOVE_TO_IDX.items()}
REAL_ACTION_SIZE = len(MOVE_TO_IDX)


# ═══════════════════════════════════════════════════════════════════════════
# BOARD ENCODING
# ═══════════════════════════════════════════════════════════════════════════

def board_to_tensor(board: chess.Board) -> np.ndarray:
    """Encode board state as (18, 8, 8) float32 numpy array."""
    state = np.zeros((TOTAL_CHANNELS, BOARD_SIZE, BOARD_SIZE), dtype=np.float32)

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece:
            ch = PIECE_TO_CH[(piece.piece_type, piece.color)]
            row = sq // 8
            col = sq % 8
            state[ch, row, col] = 1.0

    # Extra channels
    state[12] = float(board.has_kingside_castling_rights(chess.WHITE))
    state[13] = float(board.has_queenside_castling_rights(chess.WHITE))
    state[14] = float(board.has_kingside_castling_rights(chess.BLACK))
    state[15] = float(board.has_queenside_castling_rights(chess.BLACK))

    if board.ep_square is not None:
        col = board.ep_square % 8
        state[16, :, col] = 1.0

    state[17] = float(board.turn == chess.WHITE)

    return state


def legal_mask(board: chess.Board) -> np.ndarray:
    """Return boolean mask over action space for legal moves."""
    mask = np.zeros(REAL_ACTION_SIZE, dtype=np.float32)
    for move in board.legal_moves:
        uci = move.uci()
        if uci in MOVE_TO_IDX:
            mask[MOVE_TO_IDX[uci]] = 1.0
    return mask


# ═══════════════════════════════════════════════════════════════════════════
# NEURAL NETWORK
# ═══════════════════════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ChessNet(nn.Module):
    """
    Lightweight policy-value network for CPU training.
    Input : (B, 18, 8, 8)
    Output: policy logits (B, ACTION_SIZE), value (B, 1)
    """
    def __init__(self, channels=64, res_blocks=4):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv2d(TOTAL_CHANNELS, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )
        self.res_tower = nn.Sequential(*[ResBlock(channels) for _ in range(res_blocks)])

        # Policy head
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, REAL_ACTION_SIZE)
        )

        # Value head
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 4, 1, bias=False),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(4 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.input_conv(x)
        x = self.res_tower(x)
        return self.policy_head(x), self.value_head(x)


# ═══════════════════════════════════════════════════════════════════════════
# MCTS
# ═══════════════════════════════════════════════════════════════════════════

class MCTSNode:
    __slots__ = ['board', 'parent', 'move', 'children', 'visit_count',
                 'value_sum', 'prior', 'is_expanded']

    def __init__(self, board, parent=None, move=None, prior=0.0):
        self.board       = board.copy()
        self.parent      = parent
        self.move        = move
        self.children    = {}
        self.visit_count = 0
        self.value_sum   = 0.0
        self.prior       = prior
        self.is_expanded = False

    @property
    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count

    def ucb_score(self, parent_visits: int, c_puct: float = 1.4) -> float:
        u = c_puct * self.prior * math.sqrt(parent_visits) / (1 + self.visit_count)
        return self.q_value + u


class MCTS:
    def __init__(self, net: Optional['ChessNet'], device, simulations: int = 50,
                 c_puct: float = 1.4, temperature: float = 1.0):
        self.net         = net
        self.device      = device
        self.simulations = simulations
        self.c_puct      = c_puct
        self.temperature = temperature

    def _get_policy_value(self, board: chess.Board):
        """Query neural net for policy + value."""
        if self.net is None or not TORCH_AVAILABLE:
            # Fallback: uniform policy + random value
            mask = legal_mask(board)
            n = mask.sum()
            policy = mask / max(n, 1)
            value  = 0.0
            return policy, value

        state = board_to_tensor(board)
        x = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits, val = self.net(x)
        logits = logits.squeeze(0).cpu().numpy()
        val    = val.item()

        # mask illegal moves
        mask   = legal_mask(board)
        logits = np.where(mask, logits, -1e9)
        logits -= logits.max()
        probs  = np.exp(logits)
        probs  = probs * mask
        s      = probs.sum()
        policy = probs / s if s > 0 else mask / max(mask.sum(), 1)
        return policy, val

    def search(self, root_board: chess.Board) -> Dict[str, float]:
        root = MCTSNode(root_board)
        policy, _ = self._get_policy_value(root.board)
        self._expand(root, policy)

        for _ in range(self.simulations):
            node = root
            # Selection
            while node.is_expanded and node.children:
                node = max(node.children.values(),
                           key=lambda n: n.ucb_score(node.visit_count, self.c_puct)
                           if node.visit_count > 0 else random.random())

            # Terminal check
            if node.board.is_game_over():
                result = node.board.result()
                value  = 1.0 if result == "1-0" else (-1.0 if result == "0-1" else 0.0)
                if node.board.turn == chess.BLACK:
                    value = -value
            else:
                policy, value = self._get_policy_value(node.board)
                if not node.is_expanded:
                    self._expand(node, policy)

            # Backprop
            self._backprop(node, value)

        # Build move probabilities with temperature
        visits = {m: child.visit_count for m, child in root.children.items()}
        if not visits:
            return {}
        if self.temperature < 0.01:
            best = max(visits, key=visits.get)
            probs = {m: 1.0 if m == best else 0.0 for m in visits}
        else:
            total = sum(v ** (1 / self.temperature) for v in visits.values())
            probs = {m: (v ** (1 / self.temperature)) / total for m, v in visits.items()}
        return probs

    def _expand(self, node: MCTSNode, policy: np.ndarray):
        node.is_expanded = True
        for move in node.board.legal_moves:
            uci = move.uci()
            prior = policy[MOVE_TO_IDX[uci]] if uci in MOVE_TO_IDX else 0.0
            child_board = node.board.copy()
            child_board.push(move)
            node.children[uci] = MCTSNode(child_board, parent=node, move=uci, prior=prior)

    def _backprop(self, node: MCTSNode, value: float):
        while node is not None:
            node.visit_count += 1
            node.value_sum   += value
            value             = -value
            node              = node.parent


# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION HEURISTICS  (fallback + reward shaping)
# ═══════════════════════════════════════════════════════════════════════════

PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3.2,
    chess.ROOK: 5, chess.QUEEN: 9,  chess.KING: 0
}
# Piece-square tables (from White's perspective, rank 0 = rank 1)
PST_PAWN = np.array([
    [0,  0,  0,  0,  0,  0,  0,  0],
    [50, 50, 50, 50, 50, 50, 50, 50],
    [10, 10, 20, 30, 30, 20, 10, 10],
    [5,  5, 10, 25, 25, 10,  5,  5],
    [0,  0,  0, 20, 20,  0,  0,  0],
    [5, -5,-10,  0,  0,-10, -5,  5],
    [5, 10, 10,-20,-20, 10, 10,  5],
    [0,  0,  0,  0,  0,  0,  0,  0]
], dtype=np.float32) / 100.0

PST_KNIGHT = np.array([
    [-50,-40,-30,-30,-30,-30,-40,-50],
    [-40,-20,  0,  0,  0,  0,-20,-40],
    [-30,  0, 10, 15, 15, 10,  0,-30],
    [-30,  5, 15, 20, 20, 15,  5,-30],
    [-30,  0, 15, 20, 20, 15,  0,-30],
    [-30,  5, 10, 15, 15, 10,  5,-30],
    [-40,-20,  0,  5,  5,  0,-20,-40],
    [-50,-40,-30,-30,-30,-30,-40,-50]
], dtype=np.float32) / 100.0


def evaluate_board(board: chess.Board) -> float:
    """Static evaluation: positive = White advantage."""
    if board.is_checkmate():
        return -100.0 if board.turn == chess.WHITE else 100.0
    if board.is_stalemate() or board.is_insufficient_material():
        return 0.0

    score = 0.0
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if not piece:
            continue
        val   = PIECE_VALUES[piece.piece_type]
        row   = sq // 8
        col   = sq % 8
        # PST bonus
        if piece.piece_type == chess.PAWN:
            pst = PST_PAWN[row if piece.color == chess.WHITE else 7 - row, col]
        elif piece.piece_type == chess.KNIGHT:
            pst = PST_KNIGHT[row if piece.color == chess.WHITE else 7 - row, col]
        else:
            pst = 0.0
        sign  = 1.0 if piece.color == chess.WHITE else -1.0
        score += sign * (val + pst)

    # Mobility bonus
    score += 0.01 * len(list(board.legal_moves)) * (1 if board.turn == chess.WHITE else -1)
    return score


# ═══════════════════════════════════════════════════════════════════════════
# SELF-PLAY DATA GENERATION
# ═══════════════════════════════════════════════════════════════════════════

def self_play_game(mcts: MCTS, max_moves: int = 80) -> List[Tuple]:
    """Run one self-play game; return list of (state, policy, outcome)."""
    board    = chess.Board()
    history  = []

    for move_num in range(max_moves):
        if board.is_game_over():
            break

        # Reduce temperature after move 30
        mcts.temperature = 1.0 if move_num < 30 else 0.1

        probs = mcts.search(board)
        if not probs:
            break

        state = board_to_tensor(board)

        # Build policy vector
        policy_vec = np.zeros(REAL_ACTION_SIZE, dtype=np.float32)
        for uci, p in probs.items():
            if uci in MOVE_TO_IDX:
                policy_vec[MOVE_TO_IDX[uci]] = p

        # Sample move
        moves = list(probs.keys())
        ps    = np.array([probs[m] for m in moves])
        ps   /= ps.sum()
        chosen = np.random.choice(moves, p=ps)

        history.append((state, policy_vec, board.turn))
        board.push(chess.Move.from_uci(chosen))

    # Determine outcome
    result = board.result()
    if result == "1-0":
        outcome_white = 1.0
    elif result == "0-1":
        outcome_white = -1.0
    else:
        outcome_white = 0.0

    training = []
    for state, policy, turn in history:
        value = outcome_white if turn == chess.WHITE else -outcome_white
        training.append((state, policy, np.float32(value)))

    return training


# ═══════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════

def train(time_budget_minutes: float = 120.0):
    """Train the chess engine within the given CPU time budget."""
    print(f"\n{'='*60}")
    print("  DEEP RL CHESS ENGINE — TRAINING")
    print(f"  Time budget: {time_budget_minutes} min  |  Device: CPU")
    print(f"{'='*60}\n")

    if not TORCH_AVAILABLE:
        print("PyTorch not available. Skipping neural training.")
        return None

    device = torch.device("cpu")
    net    = ChessNet(channels=64, res_blocks=4).to(device)
    optimizer = optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    replay   = deque(maxlen=20000)
    mcts     = MCTS(net, device, simulations=25, temperature=1.0)

    start        = time.time()
    deadline     = start + time_budget_minutes * 60
    iteration    = 0
    total_games  = 0

    while time.time() < deadline:
        iteration += 1
        iter_start = time.time()

        # ── Self-play phase ────────────────────────────────────────────────
        games_this_iter = max(1, min(4, int((deadline - time.time()) / 45)))
        print(f"Iter {iteration:3d} | Generating {games_this_iter} self-play game(s)...")

        for _ in range(games_this_iter):
            if time.time() >= deadline:
                break
            net.eval()
            data = self_play_game(mcts, max_moves=60)
            replay.extend(data)
            total_games += 1

        if len(replay) < 128:
            continue

        # ── Training phase ─────────────────────────────────────────────────
        net.train()
        batch_size = min(256, len(replay))
        samples    = random.sample(replay, batch_size)

        states  = torch.tensor(np.array([s[0] for s in samples]), dtype=torch.float32).to(device)
        targets_p = torch.tensor(np.array([s[1] for s in samples]), dtype=torch.float32).to(device)
        targets_v = torch.tensor(np.array([s[2] for s in samples]), dtype=torch.float32).unsqueeze(1).to(device)

        for _ in range(3):   # multiple gradient steps per iteration
            policy_logits, value_pred = net(states)

            # Policy loss: cross-entropy vs MCTS visits
            log_probs  = F.log_softmax(policy_logits, dim=1)
            policy_loss = -(targets_p * log_probs).sum(dim=1).mean()

            # Value loss: MSE
            value_loss  = F.mse_loss(value_pred, targets_v)

            loss = policy_loss + value_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        elapsed  = time.time() - start
        remain   = max(0, deadline - time.time())
        print(f"         | Loss={loss.item():.4f}  "
              f"Games={total_games}  "
              f"Replay={len(replay)}  "
              f"Elapsed={elapsed/60:.1f}m  "
              f"Remaining={remain/60:.1f}m")

    # ── Save model ─────────────────────────────────────────────────────────
    torch.save(net.state_dict(), MODEL_PATH)
    total_time = (time.time() - start) / 60
    print(f"\n✓ Training complete in {total_time:.1f} min")
    print(f"  Total self-play games: {total_games}")
    print(f"  Replay buffer size:    {len(replay)}")
    print(f"  Model saved → {MODEL_PATH}\n")
    return net


# ═══════════════════════════════════════════════════════════════════════════
# ALGEBRAIC NOTATION PARSER
# ═══════════════════════════════════════════════════════════════════════════

def parse_san_to_uci(san: str, board: chess.Board) -> Optional[str]:
    """
    Convert SAN (e4, Nf3, O-O, exd5, etc.) to UCI.
    Returns UCI string or None if invalid.
    """
    san = san.strip()
    try:
        move = board.parse_san(san)
        return move.uci()
    except Exception:
        pass
    # Maybe it's already UCI
    try:
        move = chess.Move.from_uci(san)
        if move in board.legal_moves:
            return san
    except Exception:
        pass
    return None


# ═══════════════════════════════════════════════════════════════════════════
# GAME INTERFACE
# ═══════════════════════════════════════════════════════════════════════════

def display_board(board: chess.Board):
    """Pretty-print the board."""
    PIECES = {
        'P': '♙', 'N': '♘', 'B': '♗', 'R': '♖', 'Q': '♕', 'K': '♔',
        'p': '♟', 'n': '♞', 'b': '♝', 'r': '♜', 'q': '♛', 'k': '♚',
    }
    print()
    print("   a  b  c  d  e  f  g  h")
    print("  ┌──┬──┬──┬──┬──┬──┬──┬──┐")
    for rank in range(7, -1, -1):
        row = f"{rank+1} │"
        for file in range(8):
            sq    = chess.square(file, rank)
            piece = board.piece_at(sq)
            sym   = PIECES.get(piece.symbol(), '·') if piece else '·'
            bg    = '' if (rank + file) % 2 == 0 else ''
            row  += f" {sym}│"
        print(row + f" {rank+1}")
        if rank > 0:
            print("  ├──┼──┼──┼──┼──┼──┼──┼──┤")
    print("  └──┴──┴──┴──┴──┴──┴──┴──┘")
    print("   a  b  c  d  e  f  g  h")
    print()


def engine_move(board: chess.Board, net, device, simulations: int = 100) -> str:
    """Let the engine pick the best move using MCTS + policy network."""
    mcts  = MCTS(net, device, simulations=simulations, temperature=0.05)
    probs = mcts.search(board)

    if not probs:
        # Fallback: best static eval
        best_move, best_score = None, -float('inf')
        for move in board.legal_moves:
            board.push(move)
            score = -evaluate_board(board)
            board.pop()
            if score > best_score:
                best_score = score
                best_move  = move.uci()
        return best_move

    # Pick the highest-probability move (temperature ≈ 0)
    return max(probs, key=probs.get)


def play_game(net, device):
    """Interactive CLI game loop."""
    print("\n" + "="*60)
    print("  CHESS ENGINE  —  You play White, Engine plays Black")
    print("  Enter moves in SAN (e4, Nf3, O-O) or UCI (e2e4)")
    print("  Commands: 'quit' to exit, 'undo' to take back, 'hint' for suggestion")
    print("="*60)

    board = chess.Board()
    move_history = []

    while not board.is_game_over():
        display_board(board)
        print(f"Move {board.fullmove_number}  |  "
              f"{'White' if board.turn == chess.WHITE else 'Black'} to move")

        if board.is_check():
            print("  ⚠  CHECK!")

        if board.turn == chess.WHITE:
            # ── Human move ─────────────────────────────────────────────────
            while True:
                raw = input("\n  Your move: ").strip()
                if not raw:
                    continue
                if raw.lower() == 'quit':
                    print("  Goodbye!")
                    return
                if raw.lower() == 'undo':
                    if len(move_history) >= 2:
                        board.pop(); board.pop()
                        move_history = move_history[:-2]
                        print("  Took back two plies.")
                    else:
                        print("  Nothing to undo.")
                    break
                if raw.lower() == 'hint':
                    print("  Thinking...", end='', flush=True)
                    hint_uci = engine_move(board, net, device, simulations=80)
                    hint_san = board.san(chess.Move.from_uci(hint_uci))
                    print(f"\r  Hint: {hint_san}           ")
                    continue

                uci = parse_san_to_uci(raw, board)
                if uci is None:
                    print(f"  ✗ Invalid move '{raw}'. Try again.")
                    continue
                move = chess.Move.from_uci(uci)
                if move not in board.legal_moves:
                    print(f"  ✗ Illegal move '{raw}'. Try again.")
                    continue
                san = board.san(move)
                board.push(move)
                move_history.append(uci)
                print(f"  ✓ You played: {san}")
                break

        else:
            # ── Engine move ────────────────────────────────────────────────
            print("\n  Engine thinking...", end='', flush=True)
            t0      = time.time()
            uci     = engine_move(board, net, device, simulations=120)
            elapsed = time.time() - t0
            move    = chess.Move.from_uci(uci)
            san     = board.san(move)

            # Show top candidates
            mcts2  = MCTS(net, device, simulations=60, temperature=0.05)
            probs  = mcts2.search(board)
            top    = sorted(probs.items(), key=lambda x: -x[1])[:3]

            board.push(move)
            move_history.append(uci)

            # Static eval after engine move
            eval_score = evaluate_board(board)
            eval_str   = (f"+{eval_score:.2f}" if eval_score >= 0 else f"{eval_score:.2f}")

            print(f"\r  Engine played: {san:<10} ({elapsed:.1f}s)  Eval: {eval_str}")
            top_sans = []
            for m_uci, p in top[:3]:
                try:
                    b2 = board.copy()
                    b2.pop()  # temp pop to get san from pre-move board
                    # actually build sans from current position's candidate
                    top_sans.append(f"{m_uci}({p*100:.0f}%)")
                except:
                    pass
            if top:
                candidates = "  Candidates: " + "  ".join(
                    f"{u}({p*100:.0f}%)" for u, p in top[:3]
                )
                print(candidates)

    # ── Game over ──────────────────────────────────────────────────────────
    display_board(board)
    result = board.result()
    outcome = board.outcome()
    print("\n" + "="*60)
    print(f"  GAME OVER  —  Result: {result}")
    if outcome:
        if outcome.winner == chess.WHITE:
            print("  🏆 White wins!")
        elif outcome.winner == chess.BLACK:
            print("  🤖 Engine wins!")
        else:
            print("  🤝 Draw!")
    print("="*60 + "\n")


# ═══════════════════════════════════════════════════════════════════════════
# CONFIG  ←  Edit these when running in Jupyter / Colab
# ═══════════════════════════════════════════════════════════════════════════

PLAY_ONLY      = False   # True  → skip training, load saved model
                         # False → train for TRAIN_MINUTES then play
TRAIN_MINUTES  = 33      # CPU training budget in minutes
MCTS_SIMS      = 100     # MCTS simulations per engine move during play

# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def _detect_jupyter() -> bool:
    """Return True when running inside Jupyter / IPython / Colab."""
    try:
        shell = get_ipython().__class__.__name__          # noqa: F821
        return shell in ("ZMQInteractiveShell",           # Jupyter notebook/lab
                         "TerminalInteractiveShell",      # IPython terminal
                         "Shell")                         # Colab
    except NameError:
        return False


def _parse_args():
    """
    Parse CLI args when running as a plain script.
    Uses parse_known_args so Jupyter's extra --f=<kernel-json> flag
    is silently ignored instead of crashing argparse.
    """
    import argparse
    parser = argparse.ArgumentParser(description="Deep RL Chess Engine")
    parser.add_argument("--play",    action="store_true",
                        help="Skip training and load saved model")
    parser.add_argument("--minutes", type=float, default=TRAIN_MINUTES,
                        help="Training time budget in minutes (default: %(default)s)")
    parser.add_argument("--sims",    type=int,   default=MCTS_SIMS,
                        help="MCTS simulations per move during play (default: %(default)s)")
    # parse_known_args swallows unrecognised args (e.g. Jupyter's --f=...)
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[info] Ignored unrecognised arguments: {unknown}")
    return args


def main():
    # ── Resolve configuration ──────────────────────────────────────────────
    if _detect_jupyter():
        # Running in Jupyter: use the CONFIG block above, ignore sys.argv
        play_only     = PLAY_ONLY
        train_minutes = TRAIN_MINUTES
        mcts_sims     = MCTS_SIMS
        print("[Jupyter mode] Using CONFIG block settings.")
        print(f"  PLAY_ONLY={play_only}  TRAIN_MINUTES={train_minutes}  "
              f"MCTS_SIMS={mcts_sims}")
    else:
        # Running as a plain script: parse CLI flags
        args          = _parse_args()
        play_only     = args.play
        train_minutes = args.minutes
        mcts_sims     = args.sims

    device = torch.device("cpu") if TORCH_AVAILABLE else None
    net    = None

    # ── Load or train ──────────────────────────────────────────────────────
    if play_only:
        if os.path.exists(MODEL_PATH):
            print(f"Loading model from '{MODEL_PATH}'...")
            net = ChessNet(channels=64, res_blocks=4)
            net.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
            net.eval()
            print("Model loaded successfully.")
        else:
            print(f"No saved model found at '{MODEL_PATH}'.")
            print("Set PLAY_ONLY = False (or omit --play) to train first.")
            print("Continuing with random-policy engine...")
    else:
        net = train(time_budget_minutes=train_minutes)
        if net:
            net.eval()

    # ── Play ───────────────────────────────────────────────────────────────
    play_game(net, device)


if __name__ == "__main__":
    main()

In [ ]:
"""
Deep Reinforcement Learning Chess Engine
=========================================
Architecture:
  - Policy-Value Network (CNN + residual blocks)
  - MCTS with UCB1 exploration
  - Self-play training loop optimized for CPU (~35 min)
  - Algebraic notation input (e4, Nf3, O-O, etc.)

Usage (terminal / .py script):
  python chess_rl_engine.py          # train then play
  python chess_rl_engine.py --play   # load saved model and play

Usage (Jupyter / IPython / Colab):
  Edit the CONFIG block near the bottom of this file, then run all cells.
  PLAY_ONLY = False  → train then play
  PLAY_ONLY = True   → load saved model and play
"""

import os
import sys
import math
import time
import random
import pickle
import numpy as np
from collections import defaultdict, deque
from typing import Optional, Tuple, List, Dict

# ── Try importing chess (python-chess) ──────────────────────────────────────
try:
    import chess
    import chess.pgn
except ImportError:
    print("Installing python-chess...")
    os.system("pip install python-chess --quiet")
    import chess

# ── Try importing PyTorch ───────────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    print("Installing PyTorch (CPU)...")
    os.system("pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet")
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        import torch.optim as optim
        from torch.utils.data import DataLoader, TensorDataset
        TORCH_AVAILABLE = True
    except:
        TORCH_AVAILABLE = False
        print("WARNING: PyTorch unavailable, using NumPy fallback engine.")


# ═══════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════

PIECE_CHANNELS = 12        # 6 piece types × 2 colors
EXTRA_CHANNELS = 6         # castling rights × 4, en passant file, turn
TOTAL_CHANNELS = PIECE_CHANNELS + EXTRA_CHANNELS   # 18
BOARD_SIZE = 8
ACTION_SIZE = 4672         # max possible UCI moves in chess (upper bound)
MODEL_PATH = "chess_rl_model.pt"
REPLAY_PATH = "chess_replay.pkl"

PIECE_TO_CH = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}

# Build a consistent move→index mapping
def _build_move_index() -> Dict[str, int]:
    """Enumerate all possible UCI moves for 8×8 board."""
    idx = 0
    mapping = {}
    files = 'abcdefgh'
    ranks = '12345678'
    promos = ['q', 'r', 'b', 'n']
    for f1 in files:
        for r1 in ranks:
            for f2 in files:
                for r2 in ranks:
                    if f1 == f2 and r1 == r2:
                        continue
                    uci = f"{f1}{r1}{f2}{r2}"
                    mapping[uci] = idx
                    idx += 1
                    # promotions from rank 7→8 or 2→1
                    if (r1 == '7' and r2 == '8') or (r1 == '2' and r2 == '1'):
                        for p in promos:
                            mapping[uci + p] = idx
                            idx += 1
    return mapping

MOVE_TO_IDX = _build_move_index()
IDX_TO_MOVE = {v: k for k, v in MOVE_TO_IDX.items()}
REAL_ACTION_SIZE = len(MOVE_TO_IDX)


# ═══════════════════════════════════════════════════════════════════════════
# BOARD ENCODING
# ═══════════════════════════════════════════════════════════════════════════

def board_to_tensor(board: chess.Board) -> np.ndarray:
    """Encode board state as (18, 8, 8) float32 numpy array."""
    state = np.zeros((TOTAL_CHANNELS, BOARD_SIZE, BOARD_SIZE), dtype=np.float32)

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece:
            ch = PIECE_TO_CH[(piece.piece_type, piece.color)]
            row = sq // 8
            col = sq % 8
            state[ch, row, col] = 1.0

    # Extra channels
    state[12] = float(board.has_kingside_castling_rights(chess.WHITE))
    state[13] = float(board.has_queenside_castling_rights(chess.WHITE))
    state[14] = float(board.has_kingside_castling_rights(chess.BLACK))
    state[15] = float(board.has_queenside_castling_rights(chess.BLACK))

    if board.ep_square is not None:
        col = board.ep_square % 8
        state[16, :, col] = 1.0

    state[17] = float(board.turn == chess.WHITE)

    return state


def legal_mask(board: chess.Board) -> np.ndarray:
    """Return boolean mask over action space for legal moves."""
    mask = np.zeros(REAL_ACTION_SIZE, dtype=np.float32)
    for move in board.legal_moves:
        uci = move.uci()
        if uci in MOVE_TO_IDX:
            mask[MOVE_TO_IDX[uci]] = 1.0
    return mask


# ═══════════════════════════════════════════════════════════════════════════
# NEURAL NETWORK
# ═══════════════════════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ChessNet(nn.Module):
    """
    Lightweight policy-value network for CPU training.
    Input : (B, 18, 8, 8)
    Output: policy logits (B, ACTION_SIZE), value (B, 1)
    """
    def __init__(self, channels=64, res_blocks=4):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv2d(TOTAL_CHANNELS, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )
        self.res_tower = nn.Sequential(*[ResBlock(channels) for _ in range(res_blocks)])

        # Policy head
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, REAL_ACTION_SIZE)
        )

        # Value head
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 4, 1, bias=False),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(4 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.input_conv(x)
        x = self.res_tower(x)
        return self.policy_head(x), self.value_head(x)


# ═══════════════════════════════════════════════════════════════════════════
# MCTS
# ═══════════════════════════════════════════════════════════════════════════

class MCTSNode:
    __slots__ = ['board', 'parent', 'move', 'children', 'visit_count',
                 'value_sum', 'prior', 'is_expanded']

    def __init__(self, board, parent=None, move=None, prior=0.0):
        self.board       = board.copy()
        self.parent      = parent
        self.move        = move
        self.children    = {}
        self.visit_count = 0
        self.value_sum   = 0.0
        self.prior       = prior
        self.is_expanded = False

    @property
    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count

    def ucb_score(self, parent_visits: int, c_puct: float = 1.4) -> float:
        u = c_puct * self.prior * math.sqrt(parent_visits) / (1 + self.visit_count)
        return self.q_value + u


class MCTS:
    def __init__(self, net: Optional['ChessNet'], device, simulations: int = 50,
                 c_puct: float = 1.4, temperature: float = 1.0):
        self.net         = net
        self.device      = device
        self.simulations = simulations
        self.c_puct      = c_puct
        self.temperature = temperature

    def _get_policy_value(self, board: chess.Board):
        """Query neural net for policy + value."""
        if self.net is None or not TORCH_AVAILABLE:
            # Fallback: uniform policy + random value
            mask = legal_mask(board)
            n = mask.sum()
            policy = mask / max(n, 1)
            value  = 0.0
            return policy, value

        state = board_to_tensor(board)
        x = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits, val = self.net(x)
        logits = logits.squeeze(0).cpu().numpy()
        val    = val.item()

        # mask illegal moves
        mask   = legal_mask(board)
        logits = np.where(mask, logits, -1e9)
        logits -= logits.max()
        probs  = np.exp(logits)
        probs  = probs * mask
        s      = probs.sum()
        policy = probs / s if s > 0 else mask / max(mask.sum(), 1)
        return policy, val

    def search(self, root_board: chess.Board) -> Dict[str, float]:
        root = MCTSNode(root_board)
        policy, _ = self._get_policy_value(root.board)
        self._expand(root, policy)

        for _ in range(self.simulations):
            node = root
            # Selection
            while node.is_expanded and node.children:
                node = max(node.children.values(),
                           key=lambda n: n.ucb_score(node.visit_count, self.c_puct)
                           if node.visit_count > 0 else random.random())

            # Terminal check
            if node.board.is_game_over():
                result = node.board.result()
                value  = 1.0 if result == "1-0" else (-1.0 if result == "0-1" else 0.0)
                if node.board.turn == chess.BLACK:
                    value = -value
            else:
                policy, value = self._get_policy_value(node.board)
                if not node.is_expanded:
                    self._expand(node, policy)

            # Backprop
            self._backprop(node, value)

        # Build move probabilities with temperature
        visits = {m: child.visit_count for m, child in root.children.items()}
        if not visits:
            return {}
        if self.temperature < 0.01:
            best = max(visits, key=visits.get)
            probs = {m: 1.0 if m == best else 0.0 for m in visits}
        else:
            total = sum(v ** (1 / self.temperature) for v in visits.values())
            probs = {m: (v ** (1 / self.temperature)) / total for m, v in visits.items()}
        return probs

    def _expand(self, node: MCTSNode, policy: np.ndarray):
        node.is_expanded = True
        for move in node.board.legal_moves:
            uci = move.uci()
            prior = policy[MOVE_TO_IDX[uci]] if uci in MOVE_TO_IDX else 0.0
            child_board = node.board.copy()
            child_board.push(move)
            node.children[uci] = MCTSNode(child_board, parent=node, move=uci, prior=prior)

    def _backprop(self, node: MCTSNode, value: float):
        while node is not None:
            node.visit_count += 1
            node.value_sum   += value
            value             = -value
            node              = node.parent


# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION HEURISTICS  (fallback + reward shaping)
# ═══════════════════════════════════════════════════════════════════════════

PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3.2,
    chess.ROOK: 5, chess.QUEEN: 9,  chess.KING: 0
}
# Piece-square tables (from White's perspective, rank 0 = rank 1)
PST_PAWN = np.array([
    [0,  0,  0,  0,  0,  0,  0,  0],
    [50, 50, 50, 50, 50, 50, 50, 50],
    [10, 10, 20, 30, 30, 20, 10, 10],
    [5,  5, 10, 25, 25, 10,  5,  5],
    [0,  0,  0, 20, 20,  0,  0,  0],
    [5, -5,-10,  0,  0,-10, -5,  5],
    [5, 10, 10,-20,-20, 10, 10,  5],
    [0,  0,  0,  0,  0,  0,  0,  0]
], dtype=np.float32) / 100.0

PST_KNIGHT = np.array([
    [-50,-40,-30,-30,-30,-30,-40,-50],
    [-40,-20,  0,  0,  0,  0,-20,-40],
    [-30,  0, 10, 15, 15, 10,  0,-30],
    [-30,  5, 15, 20, 20, 15,  5,-30],
    [-30,  0, 15, 20, 20, 15,  0,-30],
    [-30,  5, 10, 15, 15, 10,  5,-30],
    [-40,-20,  0,  5,  5,  0,-20,-40],
    [-50,-40,-30,-30,-30,-30,-40,-50]
], dtype=np.float32) / 100.0


def evaluate_board(board: chess.Board) -> float:
    """Static evaluation: positive = White advantage."""
    if board.is_checkmate():
        return -100.0 if board.turn == chess.WHITE else 100.0
    if board.is_stalemate() or board.is_insufficient_material():
        return 0.0

    score = 0.0
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if not piece:
            continue
        val   = PIECE_VALUES[piece.piece_type]
        row   = sq // 8
        col   = sq % 8
        # PST bonus
        if piece.piece_type == chess.PAWN:
            pst = PST_PAWN[row if piece.color == chess.WHITE else 7 - row, col]
        elif piece.piece_type == chess.KNIGHT:
            pst = PST_KNIGHT[row if piece.color == chess.WHITE else 7 - row, col]
        else:
            pst = 0.0
        sign  = 1.0 if piece.color == chess.WHITE else -1.0
        score += sign * (val + pst)

    # Mobility bonus
    score += 0.01 * len(list(board.legal_moves)) * (1 if board.turn == chess.WHITE else -1)
    return score


# ═══════════════════════════════════════════════════════════════════════════
# SELF-PLAY DATA GENERATION
# ═══════════════════════════════════════════════════════════════════════════

def self_play_game(mcts: MCTS, max_moves: int = 80) -> List[Tuple]:
    """Run one self-play game; return list of (state, policy, outcome)."""
    board    = chess.Board()
    history  = []

    for move_num in range(max_moves):
        if board.is_game_over():
            break

        # Reduce temperature after move 30
        mcts.temperature = 1.0 if move_num < 30 else 0.1

        probs = mcts.search(board)
        if not probs:
            break

        state = board_to_tensor(board)

        # Build policy vector
        policy_vec = np.zeros(REAL_ACTION_SIZE, dtype=np.float32)
        for uci, p in probs.items():
            if uci in MOVE_TO_IDX:
                policy_vec[MOVE_TO_IDX[uci]] = p

        # Sample move
        moves = list(probs.keys())
        ps    = np.array([probs[m] for m in moves])
        ps   /= ps.sum()
        chosen = np.random.choice(moves, p=ps)

        history.append((state, policy_vec, board.turn))
        board.push(chess.Move.from_uci(chosen))

    # Determine outcome
    result = board.result()
    if result == "1-0":
        outcome_white = 1.0
    elif result == "0-1":
        outcome_white = -1.0
    else:
        outcome_white = 0.0

    training = []
    for state, policy, turn in history:
        value = outcome_white if turn == chess.WHITE else -outcome_white
        training.append((state, policy, np.float32(value)))

    return training


# ═══════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════

def train(time_budget_minutes: float = 120.0):
    """Train the chess engine within the given CPU time budget."""
    print(f"\n{'='*60}")
    print("  DEEP RL CHESS ENGINE — TRAINING")
    print(f"  Time budget: {time_budget_minutes} min  |  Device: CPU")
    print(f"{'='*60}\n")

    if not TORCH_AVAILABLE:
        print("PyTorch not available. Skipping neural training.")
        return None

    device = torch.device("cpu")
    net    = ChessNet(channels=64, res_blocks=4).to(device)
    optimizer = optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    replay   = deque(maxlen=20000)
    mcts     = MCTS(net, device, simulations=25, temperature=1.0)

    start        = time.time()
    deadline     = start + time_budget_minutes * 60
    iteration    = 0
    total_games  = 0

    while time.time() < deadline:
        iteration += 1
        iter_start = time.time()

        # ── Self-play phase ────────────────────────────────────────────────
        games_this_iter = max(1, min(4, int((deadline - time.time()) / 45)))
        print(f"Iter {iteration:3d} | Generating {games_this_iter} self-play game(s)...")

        for _ in range(games_this_iter):
            if time.time() >= deadline:
                break
            net.eval()
            data = self_play_game(mcts, max_moves=60)
            replay.extend(data)
            total_games += 1

        if len(replay) < 128:
            continue

        # ── Training phase ─────────────────────────────────────────────────
        net.train()
        batch_size = min(256, len(replay))
        samples    = random.sample(replay, batch_size)

        states  = torch.tensor(np.array([s[0] for s in samples]), dtype=torch.float32).to(device)
        targets_p = torch.tensor(np.array([s[1] for s in samples]), dtype=torch.float32).to(device)
        targets_v = torch.tensor(np.array([s[2] for s in samples]), dtype=torch.float32).unsqueeze(1).to(device)

        for _ in range(3):   # multiple gradient steps per iteration
            policy_logits, value_pred = net(states)

            # Policy loss: cross-entropy vs MCTS visits
            log_probs  = F.log_softmax(policy_logits, dim=1)
            policy_loss = -(targets_p * log_probs).sum(dim=1).mean()

            # Value loss: MSE
            value_loss  = F.mse_loss(value_pred, targets_v)

            loss = policy_loss + value_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        elapsed  = time.time() - start
        remain   = max(0, deadline - time.time())
        print(f"         | Loss={loss.item():.4f}  "
              f"Games={total_games}  "
              f"Replay={len(replay)}  "
              f"Elapsed={elapsed/60:.1f}m  "
              f"Remaining={remain/60:.1f}m")

    # ── Save model ─────────────────────────────────────────────────────────
    torch.save(net.state_dict(), MODEL_PATH)
    total_time = (time.time() - start) / 60
    print(f"\n✓ Training complete in {total_time:.1f} min")
    print(f"  Total self-play games: {total_games}")
    print(f"  Replay buffer size:    {len(replay)}")
    print(f"  Model saved → {MODEL_PATH}\n")
    return net


# ═══════════════════════════════════════════════════════════════════════════
# ALGEBRAIC NOTATION PARSER
# ═══════════════════════════════════════════════════════════════════════════

def parse_san_to_uci(san: str, board: chess.Board) -> Optional[str]:
    """
    Convert SAN (e4, Nf3, O-O, exd5, etc.) to UCI.
    Returns UCI string or None if invalid.
    """
    san = san.strip()
    try:
        move = board.parse_san(san)
        return move.uci()
    except Exception:
        pass
    # Maybe it's already UCI
    try:
        move = chess.Move.from_uci(san)
        if move in board.legal_moves:
            return san
    except Exception:
        pass
    return None


# ═══════════════════════════════════════════════════════════════════════════
# GAME INTERFACE
# ═══════════════════════════════════════════════════════════════════════════

def display_board(board: chess.Board):
    """Pretty-print the board."""
    PIECES = {
        'P': '♙', 'N': '♘', 'B': '♗', 'R': '♖', 'Q': '♕', 'K': '♔',
        'p': '♟', 'n': '♞', 'b': '♝', 'r': '♜', 'q': '♛', 'k': '♚',
    }
    print()
    print("   a  b  c  d  e  f  g  h")
    print("  ┌──┬──┬──┬──┬──┬──┬──┬──┐")
    for rank in range(7, -1, -1):
        row = f"{rank+1} │"
        for file in range(8):
            sq    = chess.square(file, rank)
            piece = board.piece_at(sq)
            sym   = PIECES.get(piece.symbol(), '·') if piece else '·'
            bg    = '' if (rank + file) % 2 == 0 else ''
            row  += f" {sym}│"
        print(row + f" {rank+1}")
        if rank > 0:
            print("  ├──┼──┼──┼──┼──┼──┼──┼──┤")
    print("  └──┴──┴──┴──┴──┴──┴──┴──┘")
    print("   a  b  c  d  e  f  g  h")
    print()


def engine_move(board: chess.Board, net, device, simulations: int = 100) -> str:
    """Let the engine pick the best move using MCTS + policy network."""
    mcts  = MCTS(net, device, simulations=simulations, temperature=0.05)
    probs = mcts.search(board)

    if not probs:
        # Fallback: best static eval
        best_move, best_score = None, -float('inf')
        for move in board.legal_moves:
            board.push(move)
            score = -evaluate_board(board)
            board.pop()
            if score > best_score:
                best_score = score
                best_move  = move.uci()
        return best_move

    # Pick the highest-probability move (temperature ≈ 0)
    return max(probs, key=probs.get)


def play_game(net, device):
    """Interactive CLI game loop."""
    print("\n" + "="*60)
    print("  CHESS ENGINE  —  You play White, Engine plays Black")
    print("  Enter moves in SAN (e4, Nf3, O-O) or UCI (e2e4)")
    print("  Commands: 'quit' to exit, 'undo' to take back, 'hint' for suggestion")
    print("="*60)

    board = chess.Board()
    move_history = []

    while not board.is_game_over():
        display_board(board)
        print(f"Move {board.fullmove_number}  |  "
              f"{'White' if board.turn == chess.WHITE else 'Black'} to move")

        if board.is_check():
            print("  ⚠  CHECK!")

        if board.turn == chess.WHITE:
            # ── Human move ─────────────────────────────────────────────────
            while True:
                raw = input("\n  Your move: ").strip()
                if not raw:
                    continue
                if raw.lower() == 'quit':
                    print("  Goodbye!")
                    return
                if raw.lower() == 'undo':
                    if len(move_history) >= 2:
                        board.pop(); board.pop()
                        move_history = move_history[:-2]
                        print("  Took back two plies.")
                    else:
                        print("  Nothing to undo.")
                    break
                if raw.lower() == 'hint':
                    print("  Thinking...", end='', flush=True)
                    hint_uci = engine_move(board, net, device, simulations=80)
                    hint_san = board.san(chess.Move.from_uci(hint_uci))
                    print(f"\r  Hint: {hint_san}           ")
                    continue

                uci = parse_san_to_uci(raw, board)
                if uci is None:
                    print(f"  ✗ Invalid move '{raw}'. Try again.")
                    continue
                move = chess.Move.from_uci(uci)
                if move not in board.legal_moves:
                    print(f"  ✗ Illegal move '{raw}'. Try again.")
                    continue
                san = board.san(move)
                board.push(move)
                move_history.append(uci)
                print(f"  ✓ You played: {san}")
                break

        else:
            # ── Engine move ────────────────────────────────────────────────
            print("\n  Engine thinking...", end='', flush=True)
            t0      = time.time()
            uci     = engine_move(board, net, device, simulations=120)
            elapsed = time.time() - t0
            move    = chess.Move.from_uci(uci)
            san     = board.san(move)

            # Show top candidates
            mcts2  = MCTS(net, device, simulations=60, temperature=0.05)
            probs  = mcts2.search(board)
            top    = sorted(probs.items(), key=lambda x: -x[1])[:3]

            board.push(move)
            move_history.append(uci)

            # Static eval after engine move
            eval_score = evaluate_board(board)
            eval_str   = (f"+{eval_score:.2f}" if eval_score >= 0 else f"{eval_score:.2f}")

            print(f"\r  Engine played: {san:<10} ({elapsed:.1f}s)  Eval: {eval_str}")
            top_sans = []
            for m_uci, p in top[:3]:
                try:
                    b2 = board.copy()
                    b2.pop()  # temp pop to get san from pre-move board
                    # actually build sans from current position's candidate
                    top_sans.append(f"{m_uci}({p*100:.0f}%)")
                except:
                    pass
            if top:
                candidates = "  Candidates: " + "  ".join(
                    f"{u}({p*100:.0f}%)" for u, p in top[:3]
                )
                print(candidates)

    # ── Game over ──────────────────────────────────────────────────────────
    display_board(board)
    result = board.result()
    outcome = board.outcome()
    print("\n" + "="*60)
    print(f"  GAME OVER  —  Result: {result}")
    if outcome:
        if outcome.winner == chess.WHITE:
            print("  🏆 White wins!")
        elif outcome.winner == chess.BLACK:
            print("  🤖 Engine wins!")
        else:
            print("  🤝 Draw!")
    print("="*60 + "\n")


# ═══════════════════════════════════════════════════════════════════════════
# CONFIG  ←  Edit these when running in Jupyter / Colab
# ═══════════════════════════════════════════════════════════════════════════

PLAY_ONLY      = False   # True  → skip training, load saved model
                         # False → train for TRAIN_MINUTES then play
TRAIN_MINUTES  = 33      # CPU training budget in minutes
MCTS_SIMS      = 100     # MCTS simulations per engine move during play

# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def _detect_jupyter() -> bool:
    """Return True when running inside Jupyter / IPython / Colab."""
    try:
        shell = get_ipython().__class__.__name__          # noqa: F821
        return shell in ("ZMQInteractiveShell",           # Jupyter notebook/lab
                         "TerminalInteractiveShell",      # IPython terminal
                         "Shell")                         # Colab
    except NameError:
        return False


def _parse_args():
    """
    Parse CLI args when running as a plain script.
    Uses parse_known_args so Jupyter's extra --f=<kernel-json> flag
    is silently ignored instead of crashing argparse.
    """
    import argparse
    parser = argparse.ArgumentParser(description="Deep RL Chess Engine")
    parser.add_argument("--play",    action="store_true",
                        help="Skip training and load saved model")
    parser.add_argument("--minutes", type=float, default=TRAIN_MINUTES,
                        help="Training time budget in minutes (default: %(default)s)")
    parser.add_argument("--sims",    type=int,   default=MCTS_SIMS,
                        help="MCTS simulations per move during play (default: %(default)s)")
    # parse_known_args swallows unrecognised args (e.g. Jupyter's --f=...)
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[info] Ignored unrecognised arguments: {unknown}")
    return args


def main():
    # ── Resolve configuration ──────────────────────────────────────────────
    if _detect_jupyter():
        # Running in Jupyter: use the CONFIG block above, ignore sys.argv
        play_only     = PLAY_ONLY
        train_minutes = TRAIN_MINUTES
        mcts_sims     = MCTS_SIMS
        print("[Jupyter mode] Using CONFIG block settings.")
        print(f"  PLAY_ONLY={play_only}  TRAIN_MINUTES={train_minutes}  "
              f"MCTS_SIMS={mcts_sims}")
    else:
        # Running as a plain script: parse CLI flags
        args          = _parse_args()
        play_only     = args.play
        train_minutes = args.minutes
        mcts_sims     = args.sims

    device = torch.device("cpu") if TORCH_AVAILABLE else None
    net    = None

    # ── Load or train ──────────────────────────────────────────────────────
    if play_only:
        if os.path.exists(MODEL_PATH):
            print(f"Loading model from '{MODEL_PATH}'...")
            net = ChessNet(channels=64, res_blocks=4)
            net.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
            net.eval()
            print("Model loaded successfully.")
        else:
            print(f"No saved model found at '{MODEL_PATH}'.")
            print("Set PLAY_ONLY = False (or omit --play) to train first.")
            print("Continuing with random-policy engine...")
    else:
        net = train(time_budget_minutes=train_minutes)
        if net:
            net.eval()

    # ── Play ───────────────────────────────────────────────────────────────
    play_game(net, device)


if __name__ == "__main__":
    main()